# Two-dimensional deposition in Cartesian and cylindrical grids

This notebook runs the ten-beam, $S=1/64$, no-CBET case in both coordinate systems. It compares direct ray absorption, raw deposition from independently summed capped sheets, and deposition after coherently combining the incident and reflected sheets of each beam at hydro-cell centres. It then visualizes the spatial heating and runs a coupled ray/sheet/grid convergence study.

The notebook is stored clean. The baseline pair and convergence sweep require several JAX compilations and can take several minutes on CPU.

In [ ]:
import sys
import time
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm, TwoSlopeNorm

repo_root = Path.cwd()
if not (repo_root / "configs").is_dir():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from examples._workflows import (
    PAPER_S64_CONFIGS,
    PAPER_S64_NO_CBET_ABSORPTION,
    run_simulation,
)
from pyGATH.io import load_simulation_config

plt.rcParams.update({"figure.dpi": 120, "axes.grid": False})

## Physical case and comparison quantities

Table II gives $T_e=70$ eV, $\nu_{ei,c}=206.45$ ps$^{-1}$, a 40-$\mu$m box, $I_0=1.625\times10^{14}$ W/cm$^2$, and 91.9% absorption without CBET. The density follows Eq. (13), the flow follows Eq. (33), and the super-Gaussian width is $w=460S\,\mu$m.

The paper used sixteen beams; this requested variant uses ten evenly spaced beams with the same single-beam intensity. Thus total incident power per inactive length is reduced by $10/16$, while the no-CBET absorbed fraction is unchanged because beams attenuate independently.

Three totals are kept separate:

- **Direct ray:** optical-depth loss along rays; this is the cleanest inverse-bremsstrahlung regression.
- **Raw grid:** conservative scatter of capped sheet sources, without coherent interference.
- **Coherent grid:** incident and reflected fields are combined with their phase and Maslov shift before evaluating cell heating.

In [ ]:
simulations_2d = {
    name: load_simulation_config(path) for name, path in PAPER_S64_CONFIGS.items()
}
cases_2d = {}
for geometry, simulation in simulations_2d.items():
    started = time.perf_counter()
    cases_2d[geometry] = run_simulation(simulation, return_coherent=True)
    print(f"{geometry}: {time.perf_counter() - started:.2f} s")

print("\ngeometry      direct       raw grid     coherent grid   Table II")
for geometry, (*_, checks) in cases_2d.items():
    print(
        f"{geometry:11s}  {checks['direct_absorption_fraction']:10.4%}"
        f"  {checks['deposited_absorption_fraction']:10.4%}"
        f"  {checks['coherent_deposited_absorption_fraction']:13.4%}"
        f"  {PAPER_S64_NO_CBET_ABSORPTION:8.4%}"
    )

## Deposition maps

Cylindrical cell edges are converted to Cartesian plotting coordinates so all four panels show the same physical plane. A shared logarithmic color scale makes coordinate and coherent-field differences visible.

In [ ]:
def cell_edge_xy(grid):
    if grid.geom.value == "cartesian":
        return np.meshgrid(
            np.asarray(grid.xb) * 1e6, np.asarray(grid.yb) * 1e6, indexing="ij"
        )
    radius, phi = np.meshgrid(
        np.asarray(grid.xb) * 1e6, np.asarray(grid.yb), indexing="ij"
    )
    return radius * np.cos(phi), radius * np.sin(phi)


fields = []
for trace, grid, beams, raw, coherent, checks in cases_2d.values():
    fields.extend(
        [
            np.asarray(raw.power_density)[:, :, 0],
            np.asarray(coherent.power_density)[:, :, 0],
        ]
    )
positive = np.concatenate([field[field > 0.0] for field in fields])
normalization = LogNorm(vmin=np.percentile(positive, 2), vmax=np.max(positive))

fig, axes = plt.subplots(2, 2, figsize=(10, 9), constrained_layout=True)
for column, geometry in enumerate(("cartesian", "cylindrical")):
    trace, grid, beams, raw, coherent, checks = cases_2d[geometry]
    x_edges, y_edges = cell_edge_xy(grid)
    for row, (label, deposition) in enumerate(
        (("raw capped sheets", raw), ("coherent sheets", coherent))
    ):
        image = axes[row, column].pcolormesh(
            x_edges,
            y_edges,
            np.asarray(deposition.power_density)[:, :, 0],
            shading="auto",
            norm=normalization,
            cmap="magma",
        )
        axes[row, column].set(
            title=f"{geometry}: {label}",
            xlabel="x (µm)",
            ylabel="y (µm)",
            aspect="equal",
        )
fig.colorbar(image, ax=axes, label="deposited power density (W m$^{-3}$)", shrink=0.85);

## Effect of the coherent field modifier

The maps below show $(q_{coherent}-q_{raw})/\max(q_{coherent})$. Positive and negative interference is localized mainly around overlapping incident/reflected sheets near caustics. Its spatial integral is positive for this case and recovers most of the raw field-limiter deficit.

In [ ]:
differences = {}
maximum_fraction = 0.0
for geometry, (trace, grid, beams, raw, coherent, checks) in cases_2d.items():
    raw_density = np.asarray(raw.power_density)[:, :, 0]
    coherent_density = np.asarray(coherent.power_density)[:, :, 0]
    differences[geometry] = (coherent_density - raw_density) / np.max(coherent_density)
    maximum_fraction = max(maximum_fraction, np.max(np.abs(differences[geometry])))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), constrained_layout=True)
difference_norm = TwoSlopeNorm(
    vmin=-maximum_fraction, vcenter=0.0, vmax=maximum_fraction
)
for axis, geometry in zip(axes, ("cartesian", "cylindrical"), strict=True):
    grid = cases_2d[geometry][1]
    x_edges, y_edges = cell_edge_xy(grid)
    image = axis.pcolormesh(
        x_edges,
        y_edges,
        differences[geometry],
        shading="auto",
        cmap="coolwarm",
        norm=difference_norm,
    )
    axis.set(title=geometry, xlabel="x (µm)", ylabel="y (µm)", aspect="equal")
fig.colorbar(
    image, ax=axes, label="coherent modifier / max coherent heating", shrink=0.85
);

## Radially integrated heating

A radial reduction makes it easier to compare the two coordinate systems. Cartesian cell powers are histogrammed by cell-centre radius; cylindrical powers are summed over azimuth. The result is normalized by incident power and radial-bin width.

In [ ]:
radial_edges_um = np.linspace(0.0, 20.0, 81)
radial_centres_um = 0.5 * (radial_edges_um[:-1] + radial_edges_um[1:])
radial_width_um = np.diff(radial_edges_um)


def radial_fraction_per_um(grid, deposition, incident_power):
    cell_power = np.asarray(deposition.cell_power)[:, :, 0]
    if grid.geom.value == "cartesian":
        x, y = np.meshgrid(
            np.asarray(grid.xc) * 1e6, np.asarray(grid.yc) * 1e6, indexing="ij"
        )
        radius_um = np.hypot(x, y)
    else:
        radius_um = np.broadcast_to(
            np.asarray(grid.xc)[:, None] * 1e6, cell_power.shape
        )
    binned, _ = np.histogram(
        radius_um.ravel(), bins=radial_edges_um, weights=cell_power.ravel()
    )
    return binned / incident_power / radial_width_um


fig, ax = plt.subplots(figsize=(7, 4.5), constrained_layout=True)
for geometry, (trace, grid, beams, raw, coherent, checks) in cases_2d.items():
    ax.plot(
        radial_centres_um,
        radial_fraction_per_um(grid, raw, checks["incident_power_w"]),
        "--",
        label=f"{geometry} raw",
    )
    ax.plot(
        radial_centres_um,
        radial_fraction_per_um(grid, coherent, checks["incident_power_w"]),
        label=f"{geometry} coherent",
    )
ax.set(xlabel="radius (µm)", ylabel="absorbed fraction per µm")
ax.legend(ncol=2);

## Quantified error sources

The following decomposition is deliberately not collapsed into one number:

1. **Physics/reference:** direct ray loss minus the 91.9% LPSE no-CBET value includes differences between wave and ray descriptions plus numerical trajectory error.
2. **Caustic field limiter:** raw grid minus direct ray loss measures the deficit caused mainly by capping each divergent geometrical-optics sheet and then summing sheets incoherently.
3. **Coherent reconstruction:** coherent grid minus direct ray loss retains cell-centre phase/interpolation and finite-resolution error.
4. **Coordinate representation:** Cartesian minus cylindrical results quantify hydro interpolation, domain-boundary, and native cell-geometry differences.
5. **Conservative scattering:** raw source minus deposited-inside minus outside should vanish independently of all preceding physics.
6. **Finite support:** the beam CSV represents the analytical super-Gaussian power, while ray sheets stop at the configured intensity cutoff.
7. **Resolution:** ray count controls transverse bundle geometry, sheet samples control integration along trajectories, and hydro cells control plasma interpolation and coherent phase sampling.
8. **CBET:** the regression requires zero CBET depth.

In [ ]:
for geometry, (trace, grid, beams, raw, coherent, checks) in cases_2d.items():
    direct = checks["direct_absorption_fraction"]
    raw_fraction = checks["deposited_absorption_fraction"]
    coherent_fraction = checks["coherent_deposited_absorption_fraction"]
    print(f"\n{geometry}")
    print(f"  direct - Table II:       {direct - PAPER_S64_NO_CBET_ABSORPTION:+.4%}")
    print(f"  raw grid - direct:       {raw_fraction - direct:+.4%}")
    print(f"  coherent grid - direct:  {coherent_fraction - direct:+.4%}")
    print(f"  outside / incident:      {checks['outside_power_fraction']:.3e}")
    print(
        f"  scatter residual:        {checks['deposition_conservation_error_fraction']:.3e}"
    )
    print(f"  maximum CBET depth:      {checks['maximum_cbet_depth']:.1f}")

cartesian_checks = cases_2d["cartesian"][-1]
cylindrical_checks = cases_2d["cylindrical"][-1]
for key in (
    "direct_absorption_fraction",
    "deposited_absorption_fraction",
    "coherent_deposited_absorption_fraction",
):
    print(
        f"{key}: Cartesian - cylindrical = {cartesian_checks[key] - cylindrical_checks[key]:+.4%}"
    )

## Coupled convergence sweep

The levels below increase transverse rays per beam, samples per sheet, and hydro cells together. This is an end-to-end convergence demonstration rather than a formal order study: because all three discretizations move together, the curves show whether the complete calculation stabilizes but do not assign an observed order to one component.

The cylindrical grids use half as many radial cells as azimuthal cells, matching the baseline physical spacing near the critical radius. The baseline results above are reused. The fine level is the expensive part.

In [ ]:
resolution_levels = {
    "coarse": {"rays": 12, "samples": 80, "cells_across": 80},
    "baseline": {"rays": 24, "samples": 160, "cells_across": 160},
    "fine": {"rays": 32, "samples": 240, "cells_across": 240},
}


def at_resolution(simulation, specification):
    cells = specification["cells_across"]
    ncells = (
        (cells, cells)
        if simulation.grid.geometry.value == "cartesian"
        else (cells // 2, cells)
    )
    return replace(
        simulation,
        beams=replace(simulation.beams, nrays_axis1=specification["rays"]),
        raytracing=replace(
            simulation.raytracing,
            nsamples_per_sheet=specification["samples"],
            diagnostic_samples=max(256, 2 * specification["samples"]),
        ),
        grid=replace(simulation.grid, ncells=ncells),
    )


convergence = {geometry: {} for geometry in simulations_2d}
for geometry, simulation in simulations_2d.items():
    for level, specification in resolution_levels.items():
        if level == "baseline":
            trace, grid, beams, raw, coherent, checks = cases_2d[geometry]
        else:
            started = time.perf_counter()
            varied = at_resolution(simulation, specification)
            trace, grid, beams, raw, checks = run_simulation(varied)
            print(f"{geometry} {level}: {time.perf_counter() - started:.2f} s")
        if not bool(trace.terminated):
            raise RuntimeError(f"{geometry} {level} trace did not terminate")
        convergence[geometry][level] = checks

In [ ]:
levels = list(resolution_levels)
cells_across = np.array([resolution_levels[level]["cells_across"] for level in levels])
styles = {
    "direct_absorption_fraction": ("o-", "direct ray"),
    "deposited_absorption_fraction": ("s--", "raw grid"),
    "coherent_deposited_absorption_fraction": ("^-", "coherent grid"),
}
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True, constrained_layout=True)
for axis, geometry in zip(axes, ("cartesian", "cylindrical"), strict=True):
    for key, (style, label) in styles.items():
        values = [convergence[geometry][level][key] for level in levels]
        axis.plot(cells_across, values, style, label=label)
    axis.axhline(PAPER_S64_NO_CBET_ABSORPTION, color="black", ls=":", label="Table II")
    axis.set(
        title=geometry, xlabel="cells across 40 µm box", ylabel="absorbed fraction"
    )
    axis.legend()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True, constrained_layout=True)
for axis, geometry in zip(axes, ("cartesian", "cylindrical"), strict=True):
    direct = np.array(
        [convergence[geometry][level]["direct_absorption_fraction"] for level in levels]
    )
    raw = np.array(
        [
            convergence[geometry][level]["deposited_absorption_fraction"]
            for level in levels
        ]
    )
    coherent = np.array(
        [
            convergence[geometry][level]["coherent_deposited_absorption_fraction"]
            for level in levels
        ]
    )
    axis.plot(
        cells_across,
        direct - PAPER_S64_NO_CBET_ABSORPTION,
        "o-",
        label="direct - Table II",
    )
    axis.plot(cells_across, raw - direct, "s--", label="raw - direct")
    axis.plot(cells_across, coherent - direct, "^-", label="coherent - direct")
    axis.axhline(0.0, color="black", lw=1)
    axis.set(
        title=geometry,
        xlabel="cells across 40 µm box",
        ylabel="signed absorption error",
    )
    axis.legend()
plt.show();

## Interpretation

Direct ray absorption is the inverse-bremsstrahlung implementation check and should approach the no-CBET reference. Agreement of Cartesian and cylindrical curves tests hydro interpolation and coordinate handling. The raw-to-direct gap is dominated by the deliberately capped geometrical-optics field near caustics, not by conservative cell scattering. Coherent reconstruction restores the incident/reflected cross term, but its remaining gap contains field-limiter and finite cell-phase-sampling errors.

A production energy-deposition algorithm may enforce agreement with ray energy loss, but this notebook does not renormalize either grid field: leaving the discrepancy exposed makes the validation diagnostically useful.

In [ ]:
for geometry, (trace, grid, beams, raw, coherent, checks) in cases_2d.items():
    assert bool(trace.terminated)
    assert checks["maximum_cbet_depth"] == 0.0
    assert abs(checks["deposition_conservation_error_fraction"]) < 1e-12
    assert (
        abs(checks["direct_absorption_fraction"] - PAPER_S64_NO_CBET_ABSORPTION) < 0.015
    )
    assert (
        checks["coherent_deposited_absorption_fraction"]
        > checks["deposited_absorption_fraction"]
    )
print("Baseline physics, conservation, and no-CBET checks passed.")